# 2.8 Namespaces, Scope, and Binding Rules

## Advanced / Expert-Level Python — Quantitative Research Engineering

### Overview

Python resolves identifier references through a structured execution model. A name is not simply a dictionary key found by searching from the innermost visible block outward: compilation classifies symbols, function calls create execution frames, closures retain bindings, modules provide global namespaces, and special contexts (classes, comprehensions, exceptions, annotations, and asynchronous tasks) impose additional rules.

Scope mistakes are particularly expensive in financial research: a notebook can silently depend on an earlier execution, a closure can unintentionally share a changing calibration parameter, a class can hide a global risk limit, and an asynchronous strategy can inherit the wrong research context.

This notebook combines the Python language reference with executable experiments, static compiler inspection, and a **scoped quantitative risk-policy evaluation engine** using immutable research inputs, `Decimal`, `contextvars`, and deterministic tests.

### Learning Objectives

1. Define namespaces, lexical scope, code blocks, bindings, and execution frames precisely.
2. Apply LEGB lookup while recognizing its exceptions and limitations.
3. Explain compile-time local classification, `UnboundLocalError`, and deleted bindings.
4. Distinguish `global`, `nonlocal`, mutation, and rebinding.
5. Inspect closure cells, free variables, late binding, and capture strategies.
6. Analyze comprehension scope and assignment-expression leakage.
7. Understand class-body name resolution, methods, and `__class__` cells.
8. Explain loop, context-manager, exception, and pattern-matching capture bindings.
9. Evaluate `globals()`, `locals()`, `vars()`, and explicitly provided execution namespaces.
10. Use `ast`, `symtable`, `inspect`, and `dis` to analyze bindings.
11. Recognize Python 3.12+ annotation scopes and Python 3.14 annotation changes.
12. Use `ContextVar` correctly with nested scopes, threads, and asynchronous tasks.
13. Develop a financial rule engine with isolated per-research-run context.
14. Verify context restoration, state preservation, and numerical invariants.
15. Design defensible, reproducible quantitative research software.

### Prerequisites

Sections 2.1–2.7; familiarity with functions, dictionaries, closures, and exceptions is useful.

### Environment

**Runtime:** Python 3.10+  
**Dependencies:** Standard library only  
**Financial arithmetic:** `decimal.Decimal`  
**Verification:** Assertions and `unittest`

### Research Boundary

This section explains namespaces and scope in depth. Full function protocols, packaging, inheritance, and asynchronous system architecture belong to later modules. This notebook uses those concepts only insofar as they reveal binding behavior.

### Official References

- [Python Execution Model](https://docs.python.org/3/reference/executionmodel.html)
- [Python Simple Statements](https://docs.python.org/3/reference/simple_stmts.html)
- [Python Compound Statements](https://docs.python.org/3/reference/compound_stmts.html)
- [Symbol Table](https://docs.python.org/3/library/symtable.html)
- [Context Variables](https://docs.python.org/3/library/contextvars.html)

## Notebook Roadmap

| Section | Focus |
|---|---|
| 2.8.1 | Namespaces, blocks, and execution frames |
| 2.8.2 | LEGB and shadowing |
| 2.8.3 | Compile-time local classification |
| 2.8.4 | `global`, `nonlocal`, and scope declarations |
| 2.8.5 | Closures, late binding, and closure cells |
| 2.8.6 | Comprehensions and assignment-expression scope |
| 2.8.7 | Class namespace vs. method namespace |
| 2.8.8 | Loop, exception, and pattern-binding lifetimes |
| 2.8.9 | Dynamic namespaces, `exec`, and `locals()` |
| 2.8.10 | AST, symbol tables, and bytecode |
| 2.8.11 | Version-sensitive annotation scopes |
| 2.8.12 | `ContextVar` and concurrent execution contexts |
| 2.8.13 | Advanced pitfalls and research integrity |
| 2.8.14 | Quantitative risk-context engine |
| 2.8.15 | Automated tests and numerical invariants |
| 2.8.16 | Final analytical interpretation |
| — | Advanced exercises and expert challenge |

## 2.8.1 Namespaces, Code Blocks, and Execution Frames

### Formal Concepts

- A **namespace** maps names to objects. It can be implemented using mappings, but function-local bindings need not be represented as an ordinary mutable dictionary at runtime.
- A **scope** is the textual region of a program in which a name binding can be accessed without qualification, subject to language-specific exceptions.
- A **code block** is a unit of execution and symbol analysis: module, function body, class body, and certain other constructs.
- Each function call has its own execution context; repeating a call does not reuse its local bindings as if they were module globals.
- The **module namespace** persists for the lifetime of the imported module; an interactive notebook has a similarly persistent user namespace between cell executions.
- Names are resolved, *then* their referenced objects participate in evaluation. Scope concerns the binding; mutability concerns the object.

### Nuance

Block boundaries and lexical scopes are not identical to indentation boundaries: `if`, `for`, and `with` suites generally do **not** create a new Python lexical scope. A function or comprehension usually does.

### Financial Relevance

A model feature built using a module-global calibration value may change when an unrelated cell rebinds that name, even though the feature function's source is unchanged.

In [1]:
from decimal import Decimal

GLOBAL_DISCOUNT_RATE = Decimal("0.015")

def price_discounted(amount: Decimal) -> tuple[Decimal, dict[str, object]]:
    local_amount = amount
    result = local_amount * (Decimal("1") - GLOBAL_DISCOUNT_RATE)
    return result, {key: value for key, value in locals().items()
                    if key in {"amount", "local_amount", "result"}}

first_value, first_locals = price_discounted(Decimal("1000"))
second_value, second_locals = price_discounted(Decimal("2000"))
assert first_value == Decimal("985.000")
assert second_value == Decimal("1970.000")
assert first_locals["amount"] != second_locals["amount"]

# Neither a conditional nor a loop creates an additional function scope.
if True:
    threshold_defined_in_if = Decimal("0.20")
for iteration_index in range(2):
    last_iteration_value = iteration_index
assert threshold_defined_in_if == Decimal("0.20")
assert last_iteration_value == 1
print("Independent function calls:", first_value, second_value)
print("Suite bindings:", threshold_defined_in_if, last_iteration_value)

Independent function calls: 985.000 1970.000
Suite bindings: 0.20 1


### **Analytical Interpretation**

**1. Independent frames** — Each function call gets its own local `amount`, `local_amount`, and `result` bindings; neither call overwrites the other call’s locals.

**2. Global lookup** — The function reads `GLOBAL_DISCOUNT_RATE` from the module-level namespace at call time.

**3. Suite leakage** — The names created by the top-level `if` and `for` suites remain visible in the notebook’s top-level execution namespace.

**4. Snapshot semantics** — The filtered `locals()` dictionary is created explicitly; it should not be confused with a writable handle into optimized function locals.

**5. Financial implication** — Explicit parameters make a calculation easier to reproduce than an undocumented dependency on a mutable global calibration parameter.

## 2.8.2 LEGB Lookup and Built-in Shadowing

### Lexical Resolution: A Useful Mental Model

For ordinary nested functions, lookup can be described as **L → E → G → B**:

1. **Local:** current function's bindings.
2. **Enclosing:** lexically enclosing function scopes.
3. **Global:** module namespace.
4. **Built-in:** built-in namespace available to the module.

**LEGB is not a complete language specification.** Class bodies, annotation scopes, comprehensions, and special blocks have additional rules. Lookup can fail with `NameError`; accessing a local classified at compile time but not initialized raises `UnboundLocalError` (a `NameError` subclass).

### Shadowing

A local name may hide a global or built-in with the same spelling. Assigning `sum = ...` does not alter the built-in `sum` object; it changes which binding is found first. Access `builtins.sum` for deliberate unambiguous reference.

In [2]:
import builtins

scope_label = "GLOBAL"

def make_scope_probe():
    scope_label = "ENCLOSING"
    def probe(use_local: bool):
        if use_local:
            local_label = "LOCAL"
            return local_label, scope_label, builtins.len([1, 2, 3])
        return scope_label, builtins.len([1])
    return probe

probe = make_scope_probe()
assert probe(True) == ("LOCAL", "ENCLOSING", 3)
assert probe(False) == ("ENCLOSING", 1)

# Local binding shadows a built-in only within that function.
def demonstrate_builtin_shadowing():
    sum = Decimal("9")
    try:
        sum([1, 2, 3])
    except TypeError:
        return builtins.sum([1, 2, 3])
    raise AssertionError("Expected local shadowing")

assert demonstrate_builtin_shadowing() == 6
print("Enclosing reference:", probe(False))
print("Explicit built-in reference:", demonstrate_builtin_shadowing())

Enclosing reference: ('ENCLOSING', 1)
Explicit built-in reference: 6


### **Analytical Interpretation**

**1. Enclosing lexical binding** — `probe` retains access to `scope_label` defined in its enclosing factory, not the identically named global.

**2. Local-first resolution** — `local_label` is resolved within `probe` when the relevant path binds it.

**3. Built-in shadowing** — A local decimal value named `sum` shadows the callable built-in `sum`; this does not replace `builtins.sum`.

**4. Static environment** — The textual environment of the function definition, not the caller’s local variables, determines lexical enclosing scopes.

**5. Research control** — Avoid shadowing statistical and financial functions (`sum`, `min`, `max`, `open`, `type`) in reusable research code.

## 2.8.3 Static Local Classification and UnboundLocalError

### Compile-time Symbol Classification

If a name is bound anywhere in a function body (subject to applicable `global`/`nonlocal` declarations), the compiler generally treats it as local throughout the block. This is **not** decided line-by-line at runtime.

```python
def f():
    print(config)
    config = 123
```

The read is classified as local and fails because the local binding has not yet been initialized. Similar problems occur with `del`, unreachable branches, and `except` variables that are cleared. `UnboundLocalError` is a subclass of `NameError`.

### Diagnostic Approach

Distinguish (a) unresolved global/free lookup, (b) local referenced before assignment, (c) deleted local referenced again. The runtime error can reveal the binding category but not financial correctness.

In [3]:
GLOBAL_LIMIT_283 = Decimal("0.35")

def incorrectly_shadow_global():
    try:
        return GLOBAL_LIMIT_283
    finally:
        # The compiler classifies GLOBAL_LIMIT_283 as local in this block.
        GLOBAL_LIMIT_283 = Decimal("0.40")

try:
    incorrectly_shadow_global()
except UnboundLocalError as error:
    print("Compile-time local classification:", type(error).__name__)
else:
    raise AssertionError("Expected UnboundLocalError")

def branch_does_not_define_scope(flag: bool):
    if flag:
        metric = Decimal("2")
    return metric

assert branch_does_not_define_scope(True) == Decimal("2")
try:
    branch_does_not_define_scope(False)
except UnboundLocalError:
    print("Conditionally unbound local detected")
else:
    raise AssertionError("Expected UnboundLocalError")

Compile-time local classification: UnboundLocalError
Conditionally unbound local detected


### **Analytical Interpretation**

**1. Whole-block analysis** — The compiler sees the later assignment and treats `GLOBAL_LIMIT_283` as a local name, even though the earlier read appears first.

**2. Unbound branch** — An `if` suite does not create a scope, but a path that skips its assignment leaves `metric` unbound.

**3. Exception subclass** — `UnboundLocalError` is distinct from a failed global name search, despite inheriting from `NameError`.

**4. Design response** — Pass parameters explicitly or use a deliberate scope declaration; moving the assignment lower in the function does not repair the classification.

**5. Quant failure mode** — A calibration parameter that appears to be global may unexpectedly become local after a maintenance edit, breaking production risk calculations.

## 2.8.4 `global`, `nonlocal`, and Rebinding Contracts

### Scope Directives

- `global name` declares that binding operations on `name` in the current code block refer to the module-level namespace; it is a parser directive and must obey textual restrictions.
- `nonlocal name` declares binding into the **nearest applicable enclosing function** that already binds the name; it cannot invent a missing enclosing function binding.
- Neither declaration copies objects or provides automatic synchronization.
- A nested function can **mutate** a shared mutable object without declaring `nonlocal` if it does not rebind the name; `nonlocal` is needed to rebind the enclosing name.
- A sequence of `nonlocal` assignments does not automatically form an indivisible transaction.

### Contract Design

An application-level update should validate the candidate state before publishing it. Prefer immutable snapshots and a single binding replacement for logically related fields, while documenting thread-safety separately.

In [4]:
from dataclasses import dataclass

@dataclass(frozen=True)
class CalibrationState:
    limit: Decimal
    version: int

def make_calibration(initial: Decimal):
    state = CalibrationState(initial, 0)
    def read():
        return state
    def update(candidate: Decimal):
        nonlocal state
        if not isinstance(candidate, Decimal) or not candidate.is_finite() or not (0 < candidate <= 1):
            raise ValueError("Invalid threshold")
        state = CalibrationState(candidate, state.version + 1)
        return state
    return read, update

read_calibration, update_calibration = make_calibration(Decimal("0.30"))
old_state = read_calibration()
assert update_calibration(Decimal("0.32")) == CalibrationState(Decimal("0.32"), 1)
assert old_state == CalibrationState(Decimal("0.30"), 0)
try:
    update_calibration(Decimal("1.1"))
except ValueError:
    pass
assert read_calibration().version == 1
print("Original:", old_state, "Current:", read_calibration())

# Scope syntax is invalid if there is no suitable enclosing binding.
try:
    compile("def outer():\n    def inner():\n        nonlocal missing\n", "<invalid>", "exec")
except SyntaxError:
    print("Missing nonlocal binding rejected at compilation")

Original: CalibrationState(limit=Decimal('0.30'), version=0) Current: CalibrationState(limit=Decimal('0.32'), version=1)
Missing nonlocal binding rejected at compilation


### **Analytical Interpretation**

**1. Enclosing rebinding** — `nonlocal state` enables the updater to replace the factory’s state binding.

**2. Immutable publication** — A complete immutable `CalibrationState` is constructed before one enclosing name is rebound.

**3. Previous snapshots** — `old_state` remains valid because the new state is a different immutable record.

**4. Failure preservation** — Candidate validation occurs before rebinding; a rejected threshold does not change the version.

**5. Compiler restriction** — `nonlocal` cannot refer to a name missing from all eligible enclosing function scopes.

**6. Concurrent caveat** — Single-object rebinding does not create cross-thread transaction or synchronization guarantees.

## 2.8.5 Closure Cells, Free Variables, and Late Binding

### Closure Mechanics

A nested function can retain access to bindings in an enclosing function after that outer call returns. The inner function holds **cells referencing bindings**, not necessarily frozen copies of values. Introspection tools include `function.__closure__`, `function.__code__.co_freevars`, `inspect.getclosurevars`, and `dis`.

### Late Binding in Loops

Functions constructed inside a loop often share one loop variable binding. By the time they are called, the loop has finished, so every closure may observe the last loop value.

Typical repairs:

- Bind the current value as a default parameter (`lambda x, threshold=threshold: ...`).
- Create a nested factory call for each iteration.
- Prefer immutable captured configuration objects.

**Caveat:** Default arguments capture the referenced object, not a recursive deep copy; mutating a captured mutable object remains observable.

In [5]:
import inspect

limits = [Decimal("0.10"), Decimal("0.20"), Decimal("0.30")]
late_bound = [lambda exposure: exposure <= limit for limit in limits]
early_bound = [lambda exposure, limit=limit: exposure <= limit for limit in limits]

assert [fn(Decimal("0.15")) for fn in late_bound] == [True, True, True]
assert [fn(Decimal("0.15")) for fn in early_bound] == [False, True, True]


def make_closure(limit: Decimal):
    def check(value: Decimal) -> bool:
        return value <= limit
    return check

closed = make_closure(Decimal("0.25"))
assert closed(Decimal("0.24")) is True
assert closed.__code__.co_freevars == ("limit",)
assert closed.__closure__[0].cell_contents == Decimal("0.25")
assert inspect.getclosurevars(closed).nonlocals["limit"] == Decimal("0.25")

# Capturing a mutable object is not capturing an immutable value snapshot.
mutable_limit = [Decimal("0.20")]
uses_shared_list = lambda exposure, box=mutable_limit: exposure <= box[0]
assert uses_shared_list(Decimal("0.25")) is False
mutable_limit[0] = Decimal("0.30")
assert uses_shared_list(Decimal("0.25")) is True
print("Late binding:", [fn(Decimal("0.15")) for fn in late_bound])
print("Early binding:", [fn(Decimal("0.15")) for fn in early_bound])

Late binding: [True, True, True]
Early binding: [False, True, True]


### **Analytical Interpretation**

**1. Late binding** — The functions produced by the comprehension share the comprehension’s `limit` cell and observe its final value when called.

**2. Early capture** — The default-argument form binds the current decimal object at function creation, producing independent thresholds.

**3. Closure introspection** — `co_freevars` and the corresponding closure cell reveal which enclosing binding the function references.

**4. Mutable capture** — A default argument captures a reference; later mutation of that referenced list changes the function’s result.

**5. Quant risk** — Delayed evaluation of rules constructed in a loop can silently apply the wrong threshold to every sector or instrument.

## 2.8.6 Comprehension Scopes and Assignment Expressions

### Comprehension Scope

List, set, dictionary, and generator comprehensions execute the expression/body in an implicit nested scope (implementation details vary). Their ordinary iteration variables do not leak to the containing function scope in modern Python.

### Walrus Nuance (PEP 572)

A named expression such as `(candidate := expensive_call(x))` inside a comprehension usually binds in the **containing scope**, not in the comprehension-local iteration scope. This behavior is intentionally different from normal comprehension loop variables. It has syntactic restrictions (notably around rebinding a comprehension iteration variable and certain iterable-expression positions).

### Conditional Binding

If the comprehension is empty or a short circuit skips `(name := ...)`, the name may never be bound. If the containing function has a compile-time local classification for that name, a later read can raise `UnboundLocalError`.

### Generator Timing

A generator expression defers execution of most of its body until iteration. A walrus binding may therefore occur *later* than construction of the generator.

In [6]:
# Ordinary comprehension target does not overwrite the outer name.
outer_ticker = "UNCHANGED"
selected = [ticker for ticker in ("AAPL", "MSFT") if ticker.startswith("A")]
assert selected == ["AAPL"]
assert outer_ticker == "UNCHANGED"

# A named expression targets the containing function scope.
def extract_latest_valid(prices):
    selected_prices = [price for item in prices if (price := item) > 0]
    return selected_prices, price

assert extract_latest_valid([Decimal("1"), Decimal("2")]) == (
    [Decimal("1"), Decimal("2")], Decimal("2")
)
try:
    extract_latest_valid([])
except UnboundLocalError:
    print("Empty comprehension: named-expression binding never occurred")
else:
    raise AssertionError("Expected a missing local binding")

# Generator bodies are lazy; the named-expression binding occurs when iterated.
def deferred_capture():
    generator = ((current := x * 2) for x in (1, 2))
    before = "current" in locals()
    values = list(generator)
    return before, values, current

assert deferred_capture() == (False, [2, 4], 4)
print("Generator deferred assignment:", deferred_capture())

Empty comprehension: named-expression binding never occurred
Generator deferred assignment: (False, [2, 4], 4)


### **Analytical Interpretation**

**1. Comprehension target isolation** — `ticker` belongs to the comprehension’s internal scope; it is not a new module-level binding.

**2. Walrus containing scope** — `price := item` binds `price` in the enclosing function, so it remains accessible after the comprehension finishes.

**3. Empty input** — The function classifies `price` as local but never executes the assignment expression, so reading it fails.

**4. Deferred generator** — Constructing the generator does not execute its body; iteration triggers assignments to `current`.

**5. Notebook risk** — Named expressions in top-level comprehensions can overwrite interactive notebook names, making repeated execution state-dependent.

**6. Design rule** — Use a named local result and explicit fallback for potentially empty data; avoid relying on a comprehension’s side effect for required state initialization.

## 2.8.7 Class Body Scopes, Methods, and `__class__` Cells

### Class Namespace

A class definition executes its body in a namespace that is later used to create the class object. A class body can read names defined earlier in its body.

### Method Scope

The body of an ordinary method does **not** implicitly close over the class-body namespace as a lexical enclosing function scope. Use `self.limit`, `cls.limit`, or `ClassName.limit` rather than a bare `limit` inside methods.

### Class Comprehension Caveat

Comprehensions in class bodies execute in a nested scope and usually cannot resolve arbitrary class-body names as enclosing variables. The iterable in the outermost `for` clause is a special case because it is evaluated in the enclosing context; avoid relying on this subtle distinction in professional code.

### `__class__` Closure

CPython may create a special `__class__` closure cell for methods referencing `__class__` or using zero-argument `super()`. This is not a general rule that all class attributes become free variables of methods.

In [7]:
class DeskClass:
    maximum_weight = Decimal("0.30")
    reference_from_class_body = maximum_weight

    def allowed(self, weight: Decimal) -> bool:
        return weight <= self.maximum_weight

    def declaring_class(self):
        return __class__

instance = DeskClass()
assert instance.allowed(Decimal("0.29"))
assert instance.declaring_class() is DeskClass
assert DeskClass.reference_from_class_body == Decimal("0.30")
assert "__class__" in DeskClass.declaring_class.__code__.co_freevars

# A class comprehension does not see arbitrary names in the class mapping.
try:
    class BrokenClassComprehension:
        scope_only_threshold_287 = Decimal("0.30")
        selected = [x for x in [Decimal("0.10")] if x <= scope_only_threshold_287]
except NameError:
    print("Class-comprehension lookup failed as expected")
else:
    raise AssertionError("Expected NameError for class local")
print("Method class reference:", instance.declaring_class().__name__)

Class-comprehension lookup failed as expected
Method class reference: DeskClass


### **Analytical Interpretation**

**1. Class body lookup** — `reference_from_class_body` can resolve the name `maximum_weight` earlier in the same class body.

**2. Method lookup** — `allowed()` reads the attribute through `self`; a bare class-local reference would not normally be resolved by the method’s lexical environment.

**3. Special closure cell** — `declaring_class()` uses `__class__`, triggering special closure behavior; this does not provide generic access to all class attributes.

**4. Comprehension exception** — The class comprehension cannot resolve `scope_only_threshold_287` from the class namespace in its filter expression.

**5. Engineering response** — Access configuration through explicit instance/class attributes, not implicit class-body lexical capture.

## 2.8.8 Loop, `with`, Exception, and Pattern Capture Bindings

### Non-scoping Suites

A `for` target and an `as` target in a `with` statement are bound in their containing block. Neither creates a new function-local namespace.

### Exception Handler Lifetime

The exception target in `except Exception as error` is explicitly cleared after the handler. This avoids keeping traceback/frame references alive longer than necessary. In a function, a later access may therefore raise `UnboundLocalError`.

### Structural Pattern Matching

A successful pattern may bind capture names in the surrounding local scope. The behavior of **partial bindings when a pattern fails** is not generally a portability guarantee; do not use them as a rollback mechanism. Guards can affect which case is selected, and capture bindings should not be mistaken for an immutable result object.

### API Advice

Prefer explicit returned records over relying on names incidentally bound by iteration, exceptions, or pattern matching.

In [8]:
from contextlib import nullcontext

def demonstrate_suite_bindings():
    for instrument in ("AAPL", "MSFT"):
        pass
    with nullcontext(Decimal("5")) as committed_cash:
        pass
    return instrument, committed_cash

assert demonstrate_suite_bindings() == ("MSFT", Decimal("5"))

def exception_variable_is_cleared():
    try:
        raise ValueError("bad observation")
    except ValueError as caught_error:
        assert "bad observation" in str(caught_error)
    try:
        return caught_error
    except UnboundLocalError:
        return "CLEARED"

assert exception_variable_is_cleared() == "CLEARED"

def capture_trade(record):
    match record:
        case {"ticker": str(ticker), "shares": int(shares)} if shares > 0:
            return ticker, shares
        case _:
            return None

assert capture_trade({"ticker": "AAPL", "shares": 7}) == ("AAPL", 7)
assert capture_trade({"ticker": "AAPL", "shares": -1}) is None
print("Loop / context manager:", demonstrate_suite_bindings())
print("Exception target:", exception_variable_is_cleared())
print("Matched order:", capture_trade({"ticker":"AAPL", "shares":7}))

Loop / context manager: ('MSFT', Decimal('5'))
Exception target: CLEARED
Matched order: ('AAPL', 7)


### **Analytical Interpretation**

**1. Suite target persistence** — Both `instrument` and `committed_cash` remain bound in the function after their respective suites.

**2. Exception cleanup** — `caught_error` is cleared when the handler exits, even though it was valid inside the handler.

**3. Pattern capture** — A successful case captures `ticker` and `shares` and uses them to construct an explicit returned tuple.

**4. Failed-match caution** — Do not assume a failed or partially matched pattern leaves capture bindings in a portable state.

**5. Risk engineering** — Return structured validated trade data rather than reading incidental capture names after a complex control-flow chain.

## 2.8.9 Dynamic Namespaces, `locals()`, `globals()`, and `exec`

### Namespace Introspection

- `globals()` returns the current module-global namespace dictionary.
- `locals()` reports bindings in the current local context. In optimized function scopes, use it for inspection, **not** as a portable API for rebinding Python locals. Its precise semantics have changed across versions (formalized by PEP 667 in Python 3.13).
- `vars(obj)` usually provides an object's `__dict__`; slots-only and other objects may not have one.

### Explicit Execution Namespaces

`exec(source, globals_dict, locals_mapping)` may run **trusted** dynamically produced source with explicit namespace mappings. The behavior of nested functions can depend on whether globals and locals are the same mapping.

**Security Warning:** `eval` and `exec` execute Python code. Supplying a reduced `__builtins__` mapping is **not** a reliable sandbox for untrusted code. For configurable risk rules, use a dedicated expression grammar/AST interpreter and appropriate isolation rather than arbitrary execution.

In [9]:
# This is a fixed, local, TRUSTED source literal used only for demonstration.
trusted_source = (
    "calibration = 0.25\n"
    "def approved(weight):\n"
    "    return weight <= calibration\n"
)
explicit_namespace = {"__builtins__": __builtins__}
exec(compile(trusted_source, "<trusted-example>", "exec"),
     explicit_namespace, explicit_namespace)
assert explicit_namespace["approved"](0.20) is True
assert explicit_namespace["approved"](0.30) is False
assert "calibration" in explicit_namespace

def locals_are_for_inspection():
    amount = Decimal("10")
    observed = locals()
    observed["amount"] = Decimal("99")
    return amount

assert locals_are_for_inspection() == Decimal("10")
assert "Decimal" in globals()
print("Trusted dynamic namespace:", explicit_namespace["calibration"])
print("Function local remains:", locals_are_for_inspection())

Trusted dynamic namespace: 0.25
Function local remains: 10


### **Analytical Interpretation**

**1. Explicit globals/locals** — The trusted source defines a new `approved` function in a dedicated execution mapping, separate from the notebook’s standard definitions.

**2. Global resolution in generated function** — Using the same mapping for globals and locals ensures that the function can resolve its `calibration` binding.

**3. Inspection vs. mutation** — Mutating the mapping obtained from `locals()` inside an optimized function is not a portable way to update the local variable `amount`.

**4. Security** — This example must not be repurposed to run user-supplied strategy expressions. A restricted built-in mapping alone is not a sandbox.

**5. Research policy** — Prefer compiled, reviewed modules or a validated financial expression DSL over dynamic source execution for user-configurable risk rules.

## 2.8.10 AST, `symtable`, `inspect`, and Bytecode

### Complementary Views

| Layer | Useful API | Answers |
|---|---|---|
| Syntax | `ast.parse`, `ast.walk` | Where does the source read/store names? |
| Compiler symbols | `symtable.symtable` | Is a symbol local/global/free/parameter? |
| Runtime function | `__code__`, `__closure__`, `inspect` | Which closure cells exist? |
| CPython execution | `dis` | What classes of name-loading instructions were compiled? |

**Limitations:** AST and symbol-table analysis do not reveal the identities or values of arbitrary runtime objects; bytecode is version/implementation-specific. Static analysis cannot generally prove absence of data leakage or financial mispricing.

In [10]:
import ast
import dis
import symtable

scope_source = (
    "GLOBAL = 3\n"
    "def outer(scale):\n"
    "    threshold = 2\n"
    "    def inner(value):\n"
    "        return value * scale <= threshold + GLOBAL\n"
    "    return inner\n"
)
syntax_tree = ast.parse(scope_source)
usage = [(node.id, type(node.ctx).__name__, node.lineno)
         for node in ast.walk(syntax_tree) if isinstance(node, ast.Name)]
assert ("threshold", "Store", 3) in usage

table = symtable.symtable(scope_source, "<scope-source>", "exec")
outer_table = next(ch for ch in table.get_children() if ch.get_name() == "outer")
inner_table = next(ch for ch in outer_table.get_children() if ch.get_name() == "inner")
classifications = {s.get_name(): (s.is_local(), s.is_global(), s.is_free(), s.is_parameter())
                   for s in inner_table.get_symbols()}
assert classifications["scale"][2]
assert classifications["threshold"][2]
assert classifications["GLOBAL"][1]
assert classifications["value"][3]

# Disassemble an independently defined closure, not the unexecuted source.
closure_opnames = {instruction.opname for instruction in dis.get_instructions(closed)}
assert any(op.startswith("LOAD") for op in closure_opnames)
print("Symbol classification:", classifications)
print("Closure free variables:", closed.__code__.co_freevars)
print("Bytecode operation families:", sorted(closure_opnames))

Symbol classification: {'value': (True, False, False, True), 'scale': (False, False, True, False), 'threshold': (False, False, True, False), 'GLOBAL': (False, True, False, False)}
Closure free variables: ('limit',)
Bytecode operation families: ['COMPARE_OP', 'COPY_FREE_VARS', 'LOAD_DEREF', 'LOAD_FAST', 'RESUME', 'RETURN_VALUE']


### **Analytical Interpretation**

**1. AST contexts** — The syntax tree describes source-level `Load` and `Store` roles without executing the program.

**2. Compiler classifications** — `symtable` identifies `scale` and `threshold` as free variables within `inner`, `GLOBAL` as global, and `value` as a parameter.

**3. Runtime closure cells** — `closed.__code__.co_freevars` reports the actual closure field names of an executed function.

**4. Bytecode portability** — Only instruction families are inspected; version-specific bytecode sequences are not used as correctness tests.

**5. Financial audit** — Combine static dependency maps with runtime state snapshots and independent numerical tests when auditing a model.

## 2.8.11 Annotation Scopes and Version-Specific Semantics

### Python Version Matrix

| Python version | Relevant behavior |
|---|---|
| 3.10–3.11 | No PEP 695 `type Alias = ...` syntax or `def f[T]` type-parameter syntax |
| 3.12 | **Annotation scopes** introduced for `type` statements and generic type parameters (PEP 695) |
| 3.13 | Additional annotation-scope use for type-parameter defaults (PEP 696); `locals()` semantics were clarified (PEP 667) |
| 3.14+ | Ordinary annotations are lazily evaluated by default using annotation scopes (PEP 649 / 749) |

### The Class-Scope Exception

An annotation scope immediately within a class scope can access names from the enclosing class body in ways that ordinary method and comprehension scopes cannot. Annotation scopes have their own restrictions involving `yield`, `await`, assignment expressions, and qualified names.

### Compatibility Rule

For notebooks targeting Python 3.10+, never place 3.12-only syntax in a normal code cell. Keep that source in a **string** and compile/execute it only after checking the interpreter version. The optional example below demonstrates the behavior without making older versions fail.

In [11]:
import sys

modern_source = (
    "class AnnotatedPolicy:\n"
    "    Numeric = int\n"
    "    type Series = list[Numeric]\n"
)

if sys.version_info >= (3, 12):
    versioned_namespace = {}
    exec(compile(modern_source, "<python-3.12-example>", "exec"),
         versioned_namespace, versioned_namespace)
    alias = versioned_namespace["AnnotatedPolicy"].Series
    assert alias.__value__ == list[int]
    print("Annotation-scope alias resolved:", alias.__value__)
else:
    try:
        compile(modern_source, "<version-gated-source>", "exec")
    except SyntaxError:
        print("PEP 695 syntax correctly unavailable in Python", sys.version.split()[0])
    else:
        raise AssertionError("Unexpected syntax acceptance")

Annotation-scope alias resolved: list[int]


### **Analytical Interpretation**

**1. Version guard** — The source string is compiled only when the running interpreter supports the new syntax, keeping the notebook compatible with Python 3.10+.

**2. Class annotation scope** — In the supported branch, the lazy alias resolves `Numeric` from its enclosing class namespace.

**3. Runtime documentation** — Type alias evaluation and ordinary annotation evaluation differ across releases; do not assume all annotations share one eager execution model.

**4. Static tooling** — Symbol-table APIs and scope categories also evolve; tools should record and test against supported Python versions.

**5. Financial engineering** — Library metadata must state interpreter requirements when modern type-syntax features are used in risk analytics.

## 2.8.12 Context Variables, Threads, and Asynchronous Tasks

### Problem: Request-Scoped Research State

A global mutable `CURRENT_RISK_POLICY` is unsafe when tasks process different strategies concurrently. `threading.local()` isolates some per-thread state but does not naturally express independent logical contexts within one asynchronous event loop.

### `ContextVar`

- `ContextVar.get()` reads the value in the **current execution context**.
- `.set(value)` binds a value in that context and returns a token.
- `.reset(token)` restores the prior binding for that token; use `try/finally` or a context manager.
- `copy_context()` captures the current context mapping for controlled transfer to another thread or execution context.
- `asyncio.Task` normally captures the creating context; context propagation rules can vary with task/thread creation APIs and Python versions.
- The variable isolates **bindings**, not necessarily the internal state of referenced mutable objects. Prefer immutable values.

### Design Principle

Use context variables only for appropriately scoped metadata or configuration. Do not conceal important business dependencies unnecessarily: document them and expose explicit override mechanisms.

In [12]:
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from contextvars import ContextVar, copy_context

ACTIVE_DEMO_LIMIT: ContextVar[Decimal | None] = ContextVar("ACTIVE_DEMO_LIMIT", default=None)

@contextmanager
def temporary_limit(limit: Decimal):
    token = ACTIVE_DEMO_LIMIT.set(limit)
    try:
        yield
    finally:
        ACTIVE_DEMO_LIMIT.reset(token)

assert ACTIVE_DEMO_LIMIT.get() is None
with temporary_limit(Decimal("0.30")):
    assert ACTIVE_DEMO_LIMIT.get() == Decimal("0.30")
    with temporary_limit(Decimal("0.20")):
        assert ACTIVE_DEMO_LIMIT.get() == Decimal("0.20")
    assert ACTIVE_DEMO_LIMIT.get() == Decimal("0.30")
    captured_context = copy_context()
assert ACTIVE_DEMO_LIMIT.get() is None

with ThreadPoolExecutor(max_workers=1) as pool:
    copied_result = pool.submit(captured_context.run, ACTIVE_DEMO_LIMIT.get).result()
assert copied_result == Decimal("0.30")
print("Copied context in a worker thread:", copied_result)
print("Restored caller context:", ACTIVE_DEMO_LIMIT.get())

Copied context in a worker thread: 0.30
Restored caller context: None


### **Analytical Interpretation**

**1. Context-specific binding** — Nested `temporary_limit()` invocations replace the active binding without changing other context mappings.

**2. Token restoration** — The `finally` block restores the prior value even when the body raises an exception.

**3. Explicit propagation** — `copy_context().run(...)` transfers the captured logical context into a worker thread explicitly, independent of implicit thread inheritance defaults.

**4. Immutable values** — `Decimal` values are immutable, avoiding the major hazard of using a mutable configuration object as a context variable value.

**5. Isolation boundary** — Context-based configuration is not a database transaction, authorization system, or substitute for explicit concurrency testing.

## 2.8.13 Advanced Pitfalls and Research-State Integrity

### Diagnostic Matrix

| Pitfall | Root cause | Prevention |
|---|---|---|
| Local used before initialization | Compile-time local classification | Explicit arguments, validation, early binding |
| Late-binding closure | Shared enclosing loop cell | Factory/default capture |
| Walrus leaked into enclosing scope | PEP 572 comprehension rules | Explicit intermediate result |
| Class name missing in method | Class body is not ordinary enclosing method scope | `self.attr` / `cls.attr` |
| Exception binding disappears | Automatic handler target cleanup | Return structured result |
| Notebook depends on execution history | Persistent interactive globals | Clean-kernel run |
| Concurrent strategy reads wrong policy | Shared global configuration | Immutable explicit or context-scoped policy |
| `locals()` mutation appears to fail | Function-local optimization/version behavior | Never treat `locals()` as assignment API |
| Captured mutable configuration changes later | Closure captures reference, not deep copy | Immutable snapshot |
| Untrusted `exec`/`eval` | Arbitrary code execution | Purpose-built restricted rules engine |

### Verification Standard

A successful run in one notebook kernel is not strong evidence of reproducibility. Require clean-kernel execution, deterministic reference calculations, failure-path tests, and tests that isolate competing research configurations.

In [13]:
# Wrong: a mutable global setting introduces hidden state dependency.
mutable_configuration = {"max_weight": Decimal("0.30")}

def global_dependent_check(weight: Decimal) -> bool:
    return weight <= mutable_configuration["max_weight"]

assert global_dependent_check(Decimal("0.25")) is True
mutable_configuration["max_weight"] = Decimal("0.20")
assert global_dependent_check(Decimal("0.25")) is False

# Better: capture an immutable value inside an explicitly constructed function.
def build_fixed_checker(max_weight: Decimal):
    fixed_limit = Decimal(max_weight)
    def check(weight: Decimal) -> bool:
        return weight <= fixed_limit
    return check

stable_checker = build_fixed_checker(Decimal("0.30"))
mutable_configuration["max_weight"] = Decimal("0.10")
assert stable_checker(Decimal("0.25")) is True
print("Hidden global changed; immutable closure did not")

Hidden global changed; immutable closure did not


### **Analytical Interpretation**

**1. Hidden dependency** — The same call returns different answers after an unrelated modification to a module-level dictionary.

**2. Immutable capture** — The fixed checker retains its own decimal binding and remains stable after changes to the external dictionary.

**3. Scope is not ownership** — A name can be locally bound while the object it refers to is still mutable and shared.

**4. Research reproducibility** — Capture or pass a versioned immutable policy explicitly; test outcomes against controlled snapshots.

**5. Financial audit** — Configuration provenance is as important as the numerical formula used for a portfolio limit.

## 2.8.14 Advanced Quantitative Finance Project — Scoped Risk Policy Evaluation

### Project Objective

Implement a **per-research-run risk-policy engine** that evaluates hypothetical portfolio orders under different policies without mutating the source portfolio or leaking configuration between concurrent runs.

### Architecture

```text
Immutable Portfolio Snapshot + Immutable Risk Policy
                         |
                         v
                  RiskContext(run_id)
                         |
                         v
              ContextVar scoped binding
                         |
                         v
             Pure candidate-order evaluator
                  /              \
          Approved            Rejected
                  \              /
                   Immutable Result
                         |
                         v
             Context Restoration / Tests
```

### Financial Model

For existing positions, let `shares × mark` be market value. `NAV = cash + Σ market_value`.

For a proposed BUY order: `candidate_cash = cash - (quantity × execution_price + fee)`. For a SELL order: `candidate_cash = cash + quantity × execution_price - fee`. Mark-to-market NAV after the trade uses the **existing immutable valuation marks** rather than assuming execution price equals the market mark.

For each candidate holding, `position_weight = position_market_value / candidate_NAV` and `sector_weight = sum(sector_market_values) / candidate_NAV`.

### Constraints

- Long-only; no short-selling or cash borrowing.
- No orders in instruments absent from the immutable market snapshot.
- Nonnegative cash after fees.
- Position and sector weights within policy limits.
- Order notional within a policy limit.
- Prices and monetary amounts are positive finite `Decimal` values.
- No execution or portfolio state mutation: this is a **risk evaluation** engine, not a trade settlement or backtesting system.

### Scope Guarantees

- Missing active research context raises a controlled error.
- Nested policies restore the prior context after normal exits and exceptions.
- Multiple threads explicitly evaluate independent contexts.
- Source snapshots remain unchanged.
- No global mutable current-policy dictionary is needed.

### Limitations

No market microstructure, delayed fills, tax, FX, financing, margin, shorts, market impact, or durable audit trail. Context isolation does not provide authorization or process-boundary security.

In [14]:
from contextvars import ContextVar
from contextlib import contextmanager
from dataclasses import dataclass
from decimal import Decimal
from typing import Literal

D = Decimal

def require_decimal(name: str, value: object, *, positive=False, nonnegative=False) -> None:
    if not isinstance(value, Decimal) or not value.is_finite():
        raise ValueError(f"{name} must be a finite Decimal")
    if positive and value <= 0:
        raise ValueError(f"{name} must be positive")
    if nonnegative and value < 0:
        raise ValueError(f"{name} must be nonnegative")

@dataclass(frozen=True, slots=True)
class MarkedPosition:
    ticker: str
    sector: str
    shares: int
    mark_usd: Decimal

    def __post_init__(self):
        if not self.ticker.strip() or not self.sector.strip():
            raise ValueError("Ticker and sector cannot be blank")
        if isinstance(self.shares, bool) or not isinstance(self.shares, int) or self.shares <= 0:
            raise ValueError("Shares must be a positive integer")
        require_decimal("mark_usd", self.mark_usd, positive=True)

@dataclass(frozen=True, slots=True)
class MarkedBook:
    cash_usd: Decimal
    positions: tuple[MarkedPosition, ...]

    def __post_init__(self):
        require_decimal("cash_usd", self.cash_usd, nonnegative=True)
        tickers = [position.ticker for position in self.positions]
        if len(set(tickers)) != len(tickers):
            raise ValueError("Duplicate tickers")

    @property
    def nav_usd(self) -> Decimal:
        return self.cash_usd + sum((p.shares * p.mark_usd for p in self.positions), D("0"))

@dataclass(frozen=True, slots=True)
class RiskPolicy:
    max_position_weight: Decimal
    max_sector_weight: Decimal
    max_order_notional_usd: Decimal
    transaction_fee_usd: Decimal

    def __post_init__(self):
        for name in ("max_position_weight", "max_sector_weight"):
            value = getattr(self, name)
            require_decimal(name, value, positive=True)
            if value > 1:
                raise ValueError(f"{name} cannot exceed one")
        require_decimal("max_order_notional_usd", self.max_order_notional_usd, positive=True)
        require_decimal("transaction_fee_usd", self.transaction_fee_usd, nonnegative=True)

@dataclass(frozen=True, slots=True)
class RiskContext:
    run_id: str
    book: MarkedBook
    policy: RiskPolicy

    def __post_init__(self):
        if not isinstance(self.run_id, str) or not self.run_id.strip():
            raise ValueError("A run_id is required")

ACTIVE_RISK_CONTEXT: ContextVar[RiskContext | None] = ContextVar(
    "ACTIVE_RISK_CONTEXT_2_8", default=None
)

@contextmanager
def scoped_risk_context(context: RiskContext):
    if not isinstance(context, RiskContext):
        raise TypeError("Expected RiskContext")
    token = ACTIVE_RISK_CONTEXT.set(context)
    try:
        yield context
    finally:
        ACTIVE_RISK_CONTEXT.reset(token)

def current_risk_context() -> RiskContext:
    result = ACTIVE_RISK_CONTEXT.get()
    if result is None:
        raise RuntimeError("No active research context")
    return result

print("Immutable scoped research context models initialized")

Immutable scoped research context models initialized


### **Analytical Interpretation**

**1. Strong input contracts** — Positions, books, and policy limits validate their content before entering the calculation layer.

**2. Immutable object graph** — The context contains frozen dataclasses, tuples, integers, strings, and decimal values; no mutable price or policy dictionaries are retained.

**3. ContextVar binding** — Each logical execution context has its own active `RiskContext` binding rather than a shared module-global current-policy object.

**4. Guaranteed restoration** — The context manager resets its token in `finally`, including exception exits.

**5. Financial scope** — The book records valuation marks and holdings. Evaluation will compute a hypothetical candidate without altering this source state.

**6. Limit** — Immutable dataclasses are an engineering convention, not a secure tamper-proof sandbox.

In [15]:
@dataclass(frozen=True, slots=True)
class ProposedOrder:
    ticker: str
    side: Literal["BUY", "SELL"]
    quantity: int
    execution_price_usd: Decimal

    def __post_init__(self):
        if not self.ticker.strip() or self.side not in ("BUY", "SELL"):
            raise ValueError("Invalid order identifier or side")
        if isinstance(self.quantity, bool) or not isinstance(self.quantity, int) or self.quantity <= 0:
            raise ValueError("Quantity must be a positive integer")
        require_decimal("execution_price_usd", self.execution_price_usd, positive=True)

@dataclass(frozen=True, slots=True)
class RiskDecision:
    run_id: str
    approved: bool
    violations: tuple[str, ...]
    before_nav_usd: Decimal
    candidate_nav_usd: Decimal | None
    candidate_cash_usd: Decimal | None
    candidate_position_weight: Decimal | None
    candidate_sector_weight: Decimal | None


def evaluate_scoped_order(order: ProposedOrder) -> RiskDecision:
    context = current_risk_context()
    book, policy = context.book, context.policy
    holdings = {p.ticker: p for p in book.positions}
    if order.ticker not in holdings:
        return RiskDecision(context.run_id, False, ("UNKNOWN_INSTRUMENT",),
                            book.nav_usd, None, None, None, None)
    existing = holdings[order.ticker]
    notional = order.quantity * order.execution_price_usd
    violations: list[str] = []
    if notional > policy.max_order_notional_usd:
        violations.append("ORDER_NOTIONAL_LIMIT")

    if order.side == "BUY":
        candidate_shares = existing.shares + order.quantity
        candidate_cash = book.cash_usd - notional - policy.transaction_fee_usd
        if candidate_cash < 0:
            violations.append("INSUFFICIENT_CASH")
    else:
        if order.quantity > existing.shares:
            return RiskDecision(context.run_id, False,
                                tuple(violations + ["INSUFFICIENT_SHARES"]),
                                book.nav_usd, None, None, None, None)
        candidate_shares = existing.shares - order.quantity
        candidate_cash = book.cash_usd + notional - policy.transaction_fee_usd
        if candidate_cash < 0:
            violations.append("NEGATIVE_CASH")

    candidate_positions = tuple(
        MarkedPosition(p.ticker, p.sector, candidate_shares, p.mark_usd)
        if p.ticker == order.ticker and candidate_shares > 0 else p
        for p in book.positions
        if p.ticker != order.ticker or candidate_shares > 0
    )
    candidate_nav = candidate_cash + sum(
        (p.shares * p.mark_usd for p in candidate_positions), D("0")
    )
    if candidate_nav <= 0:
        violations.append("NONPOSITIVE_NAV")
        return RiskDecision(context.run_id, False, tuple(violations),
                            book.nav_usd, candidate_nav, candidate_cash, None, None)

    position_value = D(candidate_shares) * existing.mark_usd
    sector_value = sum((p.shares * p.mark_usd for p in candidate_positions
                        if p.sector == existing.sector), D("0"))
    position_weight = position_value / candidate_nav
    sector_weight = sector_value / candidate_nav
    if position_weight > policy.max_position_weight:
        violations.append("POSITION_WEIGHT_LIMIT")
    if sector_weight > policy.max_sector_weight:
        violations.append("SECTOR_WEIGHT_LIMIT")
    return RiskDecision(
        run_id=context.run_id,
        approved=not violations,
        violations=tuple(violations),
        before_nav_usd=book.nav_usd,
        candidate_nav_usd=candidate_nav,
        candidate_cash_usd=candidate_cash,
        candidate_position_weight=position_weight,
        candidate_sector_weight=sector_weight,
    )

print("Scope-resolved pure order evaluator initialized")

Scope-resolved pure order evaluator initialized


### **Analytical Interpretation**

**1. Current context resolution** — The evaluator retrieves the `RiskContext` belonging to the calling execution context, avoiding a mutable global risk-policy singleton.

**2. Candidate construction** — Cash, positions, NAV, and weights are computed using new local objects without mutating the source book.

**3. Domain-specific checks** — Liquidity, quantity availability, order notional, position concentration, and sector concentration are evaluated separately.

**4. Mark-to-market consistency** — Post-trade NAV uses independently recorded valuation marks; it may change by the fee or by execution-vs-mark differences.

**5. Rejection semantics** — An invalid order returns structured violations; no trade is executed or settled.

**6. Precision** — All monetary calculations use finite `Decimal` values and no implicit binary-float conversion.

**7. Scope observability** — Every decision carries `run_id`, allowing investigators to connect an output to the intended research context.

In [16]:
base_book = MarkedBook(
    cash_usd=D("40000"),
    positions=(
        MarkedPosition("AAPL", "Technology", 100, D("200")),
        MarkedPosition("MSFT", "Technology", 50, D("400")),
        MarkedPosition("JPM", "Financials", 100, D("200")),
    ),
)
assert base_book.nav_usd == D("100000")

standard_policy = RiskPolicy(D("0.35"), D("0.60"), D("30000"), D("2"))
strict_policy = RiskPolicy(D("0.25"), D("0.45"), D("30000"), D("2"))
standard_context = RiskContext("RESEARCH-STANDARD", base_book, standard_policy)
strict_context = RiskContext("RESEARCH-STRICT", base_book, strict_policy)
order = ProposedOrder("AAPL", "BUY", 40, D("200"))
assert ACTIVE_RISK_CONTEXT.get() is None

with scoped_risk_context(standard_context):
    standard_result = evaluate_scoped_order(order)
    with scoped_risk_context(strict_context):
        strict_result = evaluate_scoped_order(order)
    restored_result = evaluate_scoped_order(order)

assert ACTIVE_RISK_CONTEXT.get() is None
assert standard_result.approved is True
assert strict_result.approved is False
assert strict_result.violations == ("POSITION_WEIGHT_LIMIT", "SECTOR_WEIGHT_LIMIT")
assert restored_result == standard_result
assert standard_result.candidate_cash_usd == D("31998")
assert standard_result.candidate_nav_usd == D("99998")
assert base_book.nav_usd == D("100000")

print("STANDARD:", standard_result)
print("STRICT:", strict_result)
print("RESTORED:", restored_result.run_id, restored_result.approved)

STANDARD: RiskDecision(run_id='RESEARCH-STANDARD', approved=True, violations=(), before_nav_usd=Decimal('100000'), candidate_nav_usd=Decimal('99998'), candidate_cash_usd=Decimal('31998'), candidate_position_weight=Decimal('0.2800056001120022400448008960'), candidate_sector_weight=Decimal('0.4800096001920038400768015360'))
STRICT: RiskDecision(run_id='RESEARCH-STRICT', approved=False, violations=('POSITION_WEIGHT_LIMIT', 'SECTOR_WEIGHT_LIMIT'), before_nav_usd=Decimal('100000'), candidate_nav_usd=Decimal('99998'), candidate_cash_usd=Decimal('31998'), candidate_position_weight=Decimal('0.2800056001120022400448008960'), candidate_sector_weight=Decimal('0.4800096001920038400768015360'))
RESTORED: RESEARCH-STANDARD True


### **Analytical Interpretation**

**1. Same economic order** — Both runs assess identical order inputs against the same immutable portfolio snapshot.

**2. Different policy bindings** — The standard context approves the order; the strict context rejects it for position and sector concentration.

**3. Nested restoration** — After the inner strict block exits, the outer standard binding is restored; the repeated result matches the first standard evaluation.

**4. Cash conservation** — The hypothetical purchase requires $8,000 notional plus $2 fee, leaving candidate cash of $31,998.

**5. Valuation** — The candidate NAV is $99,998 because execution equals valuation marks and the $2 fee reduces NAV.

**6. No settlement** — The source book NAV remains $100,000; the evaluator generates a risk decision, not a state transition.

In [17]:
# Explicit per-thread scoped configurations: no reliance on interpreter-specific
# implicit context inheritance or timing-sensitive thread-local state.
def scoped_worker(ctx: RiskContext, candidate: ProposedOrder) -> RiskDecision:
    with scoped_risk_context(ctx):
        return evaluate_scoped_order(candidate)

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [
        executor.submit(scoped_worker, standard_context, order),
        executor.submit(scoped_worker, strict_context, order),
    ]
    threaded_results = [future.result() for future in futures]

assert [(r.run_id, r.approved) for r in threaded_results] == [
    ("RESEARCH-STANDARD", True),
    ("RESEARCH-STRICT", False),
]
assert ACTIVE_RISK_CONTEXT.get() is None

# Exception paths must not leak an active binding.
try:
    with scoped_risk_context(strict_context):
        raise LookupError("synthetic model failure")
except LookupError:
    pass
assert ACTIVE_RISK_CONTEXT.get() is None
print("Concurrent isolated results:", [(r.run_id, r.approved) for r in threaded_results])
print("Context clean after raised exception:", ACTIVE_RISK_CONTEXT.get())

Concurrent isolated results: [('RESEARCH-STANDARD', True), ('RESEARCH-STRICT', False)]
Context clean after raised exception: None


### **Analytical Interpretation**

**1. Per-thread activation** — Each worker activates its own research context before evaluating the same order.

**2. No cross-contamination** — The returned policy-specific decisions are independent, regardless of worker scheduling.

**3. Exception-safe scope** — A deliberate exception inside a scoped block still triggers token reset.

**4. Caller isolation** — The main notebook context remains unbound after all workers finish.

**5. Production boundary** — This tests logical context isolation, not market connectivity, thread-safe settlement, authorization, or durable storage.

In [18]:
# ============================================================
# ASYNCIO TASK-CREATION CONTEXT PROPAGATION
# ============================================================

import asyncio

async def deferred_async_risk_evaluation() -> RiskDecision:
    """Inspect the active policy after yielding to the event loop."""
    await asyncio.sleep(0)
    return evaluate_scoped_order(order)

async def create_independent_research_tasks():
    """Create tasks while two different contexts are active."""
    with scoped_risk_context(standard_context):
        standard_task = asyncio.create_task(deferred_async_risk_evaluation())

    with scoped_risk_context(strict_context):
        strict_task = asyncio.create_task(deferred_async_risk_evaluation())

    return await asyncio.gather(standard_task, strict_task)

# Notebook kernels already have an active asyncio event loop.
# Run a separate loop inside a worker thread to keep this cell
# executable in both JupyterLab and ordinary Python scripts.
with ThreadPoolExecutor(max_workers=1) as executor:
    async_results = executor.submit(
        asyncio.run, create_independent_research_tasks()
    ).result()

assert [(r.run_id, r.approved) for r in async_results] == [
    ("RESEARCH-STANDARD", True),
    ("RESEARCH-STRICT", False),
]
assert ACTIVE_RISK_CONTEXT.get() is None
print("Async task contexts:", [(r.run_id, r.approved) for r in async_results])

Async task contexts: [('RESEARCH-STANDARD', True), ('RESEARCH-STRICT', False)]


### **Analytical Interpretation**

**1. Creation-Time Context Capture** — Each `asyncio.Task` inherits the logical context active when `asyncio.create_task()` is invoked, even though the coroutine body runs later.

**2. Deferred Resolution** — `deferred_async_risk_evaluation()` yields once before reading the active research context; its result still reflects the correct task-specific policy.

**3. Independent Outcomes** — The same order is approved under the standard policy and rejected under the strict policy, without a shared mutable global `CURRENT_POLICY`.

**4. Context Restoration** — Leaving the context manager in the orchestrating coroutine restores its previous binding, while already-created tasks retain their independently captured contexts.

**5. Notebook Execution** — The demonstration uses a separate worker thread to run a new event loop, avoiding a nested `asyncio.run()` call within JupyterLab's existing event loop.

**6. Security and Ownership Limitation** — Context propagation isolates bindings but would not isolate shared mutable objects referenced by those bindings; immutable context payloads are essential to this example.

## 2.8.15 Automated Testing and Numerical Invariants

### Invariants

1. No active context ⇒ a controlled `RuntimeError`.
2. Entering a context changes the current binding only for that context.
3. Leaving a context, including by exception, restores its previous binding.
4. A rejected order leaves the source book and policy unchanged.
5. A position or sector weight limit is tested using **candidate NAV**, not starting NAV.
6. Price and cash calculations use finite `Decimal` values throughout.
7. A standard and strict policy can produce different decisions on identical market inputs.
8. Every output carries the active research `run_id`.
9. Missing instruments and impossible sales are rejected deterministically.
10. Thread workers activate explicitly independent contexts rather than depending on thread inheritance defaults.
11. Scope-introspection results are compiler-version aware.
12. Financial accounting checks are analytically independent of a particular print format.

The following tests exercise both normal and failure paths. They do not prove full process isolation, distributed consistency, or error-free market execution.

In [19]:
import unittest

class ScopedRiskResearchTests(unittest.TestCase):
    def test_no_context_rejected(self):
        with self.assertRaises(RuntimeError):
            evaluate_scoped_order(order)

    def test_book_nav_reference(self):
        self.assertEqual(base_book.nav_usd, D("100000"))

    def test_standard_approval(self):
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(order)
        self.assertTrue(result.approved)

    def test_strict_rejection(self):
        with scoped_risk_context(strict_context):
            result = evaluate_scoped_order(order)
        self.assertEqual(result.violations, ("POSITION_WEIGHT_LIMIT", "SECTOR_WEIGHT_LIMIT"))

    def test_context_restoration(self):
        with scoped_risk_context(standard_context):
            with scoped_risk_context(strict_context):
                self.assertIs(current_risk_context(), strict_context)
            self.assertIs(current_risk_context(), standard_context)
        self.assertIsNone(ACTIVE_RISK_CONTEXT.get())

    def test_context_restoration_after_error(self):
        with self.assertRaises(LookupError):
            with scoped_risk_context(strict_context):
                raise LookupError("test")
        self.assertIsNone(ACTIVE_RISK_CONTEXT.get())

    def test_candidate_cash(self):
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(order)
        self.assertEqual(result.candidate_cash_usd, D("31998"))

    def test_candidate_nav(self):
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(order)
        self.assertEqual(result.candidate_nav_usd, D("99998"))

    def test_candidate_position_weight(self):
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(order)
        self.assertEqual(result.candidate_position_weight, D("28000") / D("99998"))

    def test_candidate_sector_weight(self):
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(order)
        self.assertEqual(result.candidate_sector_weight, D("48000") / D("99998"))

    def test_source_not_mutated(self):
        original = base_book
        with scoped_risk_context(standard_context):
            evaluate_scoped_order(order)
        self.assertIs(original, base_book)
        self.assertEqual(original.nav_usd, D("100000"))
        self.assertEqual(original.positions[0].shares, 100)

    def test_order_notional_limit(self):
        massive = ProposedOrder("AAPL", "BUY", 160, D("200"))
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(massive)
        self.assertIn("ORDER_NOTIONAL_LIMIT", result.violations)

    def test_insufficient_cash(self):
        costly = ProposedOrder("JPM", "BUY", 210, D("200"))
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(costly)
        self.assertIn("INSUFFICIENT_CASH", result.violations)

    def test_insufficient_shares(self):
        oversell = ProposedOrder("MSFT", "SELL", 51, D("400"))
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(oversell)
        self.assertEqual(result.violations, ("INSUFFICIENT_SHARES",))

    def test_unknown_instrument(self):
        unknown = ProposedOrder("NVDA", "BUY", 1, D("100"))
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(unknown)
        self.assertEqual(result.violations, ("UNKNOWN_INSTRUMENT",))

    def test_exact_full_exit(self):
        exit_all = ProposedOrder("MSFT", "SELL", 50, D("400"))
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(exit_all)
        self.assertTrue(result.approved)
        self.assertEqual(result.candidate_position_weight, D("0"))

    def test_fee_reduces_nav(self):
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(order)
        self.assertEqual(result.before_nav_usd - result.candidate_nav_usd, D("2"))

    def test_non_decimal_prices_rejected(self):
        with self.assertRaises(ValueError):
            MarkedPosition("BAD", "Tech", 1, 1.5)

    def test_boolean_shares_rejected(self):
        with self.assertRaises(ValueError):
            MarkedPosition("BAD", "Tech", True, D("10"))

    def test_duplicate_tickers_rejected(self):
        with self.assertRaises(ValueError):
            MarkedBook(D("0"), (base_book.positions[0], base_book.positions[0]))

    def test_nonfinite_policy_rejected(self):
        with self.assertRaises(ValueError):
            RiskPolicy(D("NaN"), D("0.5"), D("100"), D("1"))

    def test_empty_context_id_rejected(self):
        with self.assertRaises(ValueError):
            RiskContext("", base_book, standard_policy)

    def test_run_id_provenance(self):
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(order)
        self.assertEqual(result.run_id, "RESEARCH-STANDARD")

    def test_sequential_reproducibility(self):
        with scoped_risk_context(standard_context):
            one = evaluate_scoped_order(order)
            two = evaluate_scoped_order(order)
        self.assertEqual(one, two)

    def test_thread_isolation(self):
        with ThreadPoolExecutor(max_workers=2) as executor:
            fst = executor.submit(scoped_worker, standard_context, order)
            snd = executor.submit(scoped_worker, strict_context, order)
            first, second = fst.result(), snd.result()
        self.assertTrue(first.approved)
        self.assertFalse(second.approved)
        self.assertIsNone(ACTIVE_RISK_CONTEXT.get())

    def test_nested_snapshot_identity(self):
        with scoped_risk_context(standard_context):
            self.assertIs(current_risk_context().book, base_book)
            with scoped_risk_context(strict_context):
                self.assertIs(current_risk_context().book, base_book)
            self.assertIs(current_risk_context().policy, standard_policy)

    def test_no_policy_leak_after_unknown_order(self):
        with scoped_risk_context(strict_context):
            evaluate_scoped_order(ProposedOrder("XXX", "BUY", 1, D("1")))
        self.assertIsNone(ACTIVE_RISK_CONTEXT.get())

    def test_very_large_price_stays_decimal(self):
        value = D("12345678901234567890.123456789")
        high = MarkedPosition("BIG", "Other", 1, value)
        self.assertEqual(high.mark_usd, value)

    def test_strict_sector_overflow(self):
        sector_tight = RiskContext("SECTOR", base_book,
                                   RiskPolicy(D("0.50"), D("0.45"), D("30000"), D("2")))
        with scoped_risk_context(sector_tight):
            result = evaluate_scoped_order(order)
        self.assertEqual(result.violations, ("SECTOR_WEIGHT_LIMIT",))

    def test_sell_reduces_technology_weight(self):
        sale = ProposedOrder("AAPL", "SELL", 20, D("200"))
        with scoped_risk_context(standard_context):
            result = evaluate_scoped_order(sale)
        self.assertTrue(result.approved)
        self.assertLess(result.candidate_sector_weight, D("0.40"))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(ScopedRiskResearchTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()
print("Validated test count:", result.testsRun)

.

.

.

.

.

.

... (22 outputs left out)

.

.

----------------------------------------------------------------------
Ran 30 tests in 0.011s

OK


Validated test count: 30


### **Analytical Interpretation**

**1. Scope contracts** — Tests confirm missing-context errors, nested scope restoration, error-path reset, and independent thread bindings.

**2. Pure evaluation** — The original position objects and cash remain unchanged after both approvals and rejections.

**3. Financial arithmetic** — Reference candidate cash, NAV, sector weights, and position weights are checked against separately specified exact Decimal expressions.

**4. Boundary handling** — Unknown symbols, impossible sales, insufficient cash, excessive notional, invalid prices, and malformed context IDs are rejected.

**5. Determinism** — Repeated evaluations of identical inputs produce equal immutable result records.

**6. Testing limit** — The tests do not claim safe concurrent trade settlement or statistical predictive power; only context and hypothetical risk-evaluation properties are exercised.

## 2.8.16 Final Analytical Interpretation

### **Analytical Interpretation**

**1. The Core Object–Name Separation.** A scope determines *which binding* a name refers to; object mutability determines whether shared references can observe the same state changes.

**2. Compile-Time Scope.** Local/global/free classifications are determined by the compiler for many constructs, not by whether a name has already appeared on a runtime execution path.

**3. LEGB and Exceptions.** LEGB is a sound intuition for ordinary nested functions, but class bodies and annotation scopes require more precise rules.

**4. Closures Are Bindings, Not Automatically Snapshots.** A closure referencing a changing cell behaves differently from one using a captured immutable default value; mutable captured objects remain a contamination risk.

**5. Comprehension Boundaries.** Iteration targets have their own comprehension scope, whereas named expressions may bind in the containing scope. Generator evaluation can defer assignment.

**6. Exception Lifetime.** `except ... as error` targets are cleared at handler exit; error-handling code must not assume a durable exception binding.

**7. Static Analysis.** `ast`, `symtable`, closure metadata, and disassembly describe different aspects of source structure, symbol classification, and runtime function layout. No one tool proves all possible state dependencies.

**8. Python Version Evolution.** Annotation scopes and annotation evaluation semantics changed notably in Python 3.12–3.14. A credible Python portfolio documents version requirements and avoids incompatible source syntax.

**9. Context Isolation.** `ContextVar` separates logical execution-context bindings. It does not isolate the internal state of mutable objects or provide authorization or transaction semantics.

**10. Scoped Financial Policy.** The same order yields different outcomes under strict and standard policies without modifying either the historical book or a shared global configuration object.

**11. Numerical Integrity.** The risk evaluator applies `Decimal`-based accounting, after-fee candidate NAV, and post-trade position/sector constraints. The calculations remain hypothetical; they are not executions.

**12. Failure Recovery.** Explicit token reset and immutable research contexts prevent common configuration-leakage failures in nested and concurrent evaluation.

**13. Research Reproducibility.** Stable inputs, explicit context provenance, independent tests, and clean-kernel execution are necessary complements to correct Python name resolution.

**14. Engineering Boundary.** Scope correctness does not, by itself, establish statistical validity, model calibration integrity, market data point-in-time correctness, or production reliability.

### Final Assessment

Understanding Python namespaces means reasoning about compiler decisions, lexical visibility, runtime binding, object references, execution contexts, and the exceptional rules of different code blocks. In professional quantitative research, these semantics must be converted into **explicit, testable policies for state ownership and reproducibility**.

## Advanced Practice Exercises — Independent Work

### Exercise 1 — Scope-Aware Static Dependency Scanner

Write a scanner that accepts a Python module as source text and reports every external reference for each function. Combine AST usage with `symtable`; classify local, free, module-global, and built-in names. Report nested scopes accurately. **Challenge:** detect undocumented global-risk-parameter dependencies without executing the source. Explain why name resolution alone does not prove data independence.

### Exercise 2 — Compiler-Determined Local Errors

Construct at least eight examples where a name is classified local but not bound on the executed path. Include conditional branches, `del`, `try/except`, `finally`, pattern captures, and comprehensions. Verify the exact exception **class** without relying on exact error messages. Classify each example as safe/unsafe to refactor.

### Exercise 3 — Closure Calibration and Late Binding

Create 50 different factor filters in a loop. Demonstrate the late-binding bug, then repair using (a) default-value capture, (b) a factory function, and (c) an immutable configuration dataclass. Add regression tests for every factor. Evaluate memory and call overhead without extrapolating one microbenchmark to all workloads.

### Exercise 4 — Comprehension and Walrus Binding Audit

Demonstrate iteration-target isolation, containing-scope named-expression binding, generator laziness, and empty-input `UnboundLocalError`. Inspect `co_freevars` and symbol tables. Explain why relying on walrus side effects is fragile in data-quality pipelines.

### Exercise 5 — Class Scope vs. Annotation Scope

Build version-gated examples for ordinary class attributes, methods, class comprehensions, `__class__` cells, and PEP 695 `type` aliases. Produce a compatibility report for 3.10, 3.11, 3.12, 3.13, and 3.14+. Never use unsupported syntax in the notebook's normal code cells.

### Exercise 6 — Exception-Target Lifecycle

Create a function that retains exception details safely without retaining the original exception/traceback object unintentionally. Compare string summaries, structured immutable error records, and captured traceback objects. Document any memory-retention trade-offs.

### Exercise 7 — Notebook Hidden-State Audit

Build a reproducibility protocol that uses a fresh kernel, executes cells in order, records dependency versions, and catches references that exist only because a user ran cells out of order. Explain the difference between a notebook that passes interactively and one that passes as a clean-build artifact.

### Exercise 8 — ContextVar vs. Thread-Local Isolation

Implement one per-task context using `ContextVar` and one per-thread context using `threading.local()`. Demonstrate different behavior for many concurrent tasks within one event loop. Use deterministic barriers or explicit events where needed; do not depend on lucky scheduling.

### Exercise 9 — Asynchronous Risk Context Propagation

Create separate `asyncio.Task` instances for two research policies. Test scope inheritance at task creation, restoration after cancellation, and behavior under nested context overrides. Keep policy objects immutable. Include an explanation of why `ContextVar` isolation is not a transaction system.

### Exercise 10 — Scoped Portfolio Validation

Extend the project with optional liquidity and volatility limits. Keep the evaluation pure: reject unknown market data, compute hypothetical post-trade NAV, produce structured violations, and never mutate the source book. Include independently computed reference values.

### Exercise 11 — Cross-Run Configuration Leakage Test

Run thousands of independent policy evaluations across threads and asynchronous tasks using deterministic seeds. Prove every result carries the correct run ID and policy. Inject exceptions and verify context restoration. Clarify what the tests cannot prove about scheduling and process isolation.

### Exercise 12 — Source-Code Security Review

Build a restricted configuration expression language using a validated AST grammar rather than unrestricted `eval` or `exec`. Support only named financial inputs, finite numeric constants, comparisons, and Boolean operations. Enforce syntax and resource limits, document limitations, and construct adversarial regression tests.

## Expert Challenge — Reproducible Multi-Strategy Research Context Platform

### Objective

Design a research platform in which thousands of independently configured quantitative strategies can evaluate portfolio risk concurrently **without accidental configuration leakage or mutation of shared observations**.

### Required Components

1. **Namespace dependency auditor:** AST and symbol-table-based reports of free/global names, nested bindings, and suspicious shadowing.
2. **Immutable research context:** versioned market snapshot, policy, run ID, model version, and valuation timestamp with explicit provenance.
3. **Scoped policy manager:** `ContextVar` with nested overrides, exception-safe resets, explicit thread propagation, and task-level tests.
4. **Pure risk evaluator:** deterministic order decisions using exact financial units, transaction costs, concentration, and liquidity constraints.
5. **No hidden state:** documented inputs, local calculations, immutable returns, and independent snapshot construction.
6. **Reproducibility:** replay of evaluations from versioned inputs without reliance on notebook execution history.
7. **Correctness:** data validation, independently derived accounting invariants, failure-injection tests, and version-sensitive scope tests.
8. **Security boundary:** no arbitrary execution of external strategy source; use a reviewed restricted rules grammar.
9. **Quality engineering:** typed APIs, linting, CI, code-coverage reports, structured diagnostics, and dependency pinning.
10. **Performance:** profiling of context capture, immutable-object construction, and symbol analysis without ungrounded speed claims.

### Evaluation Criteria

| Dimension | Weight |
|---|---:|
| Language and scope correctness | 25% |
| Context isolation and state integrity | 25% |
| Financial arithmetic and constraints | 20% |
| Testing and failure recovery | 15% |
| Architecture and reproducibility | 10% |
| Documentation | 5% |

### Deliverables

- Runnable Python package and Jupyter research demonstration.
- Full test suite covering nested scopes, exceptions, threads, tasks, and version differences.
- AST/symbol-table dependency audit.
- Immutable financial data schemas and risk-decision reports.
- Deterministic replay and research provenance record.
- Analysis of limitations: context isolation, process security, database transactions, and portfolio execution modeling are distinct problems.

### Completion Standard

Demonstrate with explicit tests that two concurrently evaluated strategies using identical order and market inputs can employ independent policies and produce correct, reproducible, auditable decisions without modifying shared financial state.